# PromptOpt Stage C: LoRA training (Colab, T4 GPU)

Fine-tunes **Qwen2.5-0.5B-Instruct** with LoRA to fill the IR fields Stage B left unresolved (task, output_format,
constraints, category), as JSON. Plan, data format and evaluation: `docs/STAGE_C_PLAN.md`.

**Before running:** upload `stage_c_data_v1.zip` to `MyDrive/PromptOpt/stage_c/` and pick *Runtime > Change runtime
type > T4 GPU*. Then *Runtime > Run all*. About 30-45 min on a T4.

**Output:** `MyDrive/PromptOpt/stage_c/outputs/stage_c_adapter_v1.zip` (the best adapter by val loss, the training
summary and the val predictions). That zip is what goes back into the repo.

All training code is in `train_stage_c.py` inside the zip (the same file `backend/app/stage_c/train.py` runs locally),
so Colab and local runs are identical. Settings come from `config.json`: seed 13, max length 512, LoRA r=16/alpha=32 on
all linear projections, lr 2e-4, effective batch 16, up to 3 epochs, early stopping on val loss (patience 3 evals of
50 steps), fp16 on T4, greedy decoding for evaluation.

In [ ]:
# Paths (change only if you put the zip somewhere else)
DRIVE_DIR = "/content/drive/MyDrive/PromptOpt/stage_c"
ZIP_PATH = f"{DRIVE_DIR}/stage_c_data_v1.zip"
OUT_DRIVE = f"{DRIVE_DIR}/outputs"
DATA = "/content/stage_c_data_v1"          # unzipped here
OUT = "/content/stage_c_out"               # adapter written here, then zipped to Drive
EVAL_N = 60                                # val examples for the zero-shot vs LoRA comparison (all 10 routed + 50 forced)

In [ ]:
!pip -q install "transformers>=4.46" "peft>=0.13" accelerate
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > T4 GPU, then run again"
print(torch.__version__, torch.cuda.get_device_name(0))

## 1. Mount Drive, unzip, verify sha256

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os, zipfile
assert os.path.exists(ZIP_PATH), f"Upload stage_c_data_v1.zip to {DRIVE_DIR}/ first"
with zipfile.ZipFile(ZIP_PATH) as z:
    z.extractall("/content")
print(sorted(os.listdir(DATA)))

In [ ]:
import sys, json
from pathlib import Path
sys.path.insert(0, DATA)
import train_stage_c as sc

manifest = sc.verify(Path(DATA))           # stops here if any file's sha256 differs from manifest.json
cfg = json.loads(Path(DATA, "config.json").read_text())
print(json.dumps(manifest["examples"], indent=1))
print({k: cfg[k] for k in ("base_model", "seed", "max_length")}, cfg["train"])

## 2. Data check: lengths under the chat template

In [ ]:
from transformers import AutoTokenizer
from collections import Counter
tok = AutoTokenizer.from_pretrained(cfg["base_model"])
train_rows = sc.load_jsonl(Path(DATA, "train.jsonl"))
lens = sorted(len(tok(tok.apply_chat_template(r["messages"], tokenize=False))["input_ids"]) for r in train_rows)
q = lambda p: lens[min(len(lens) - 1, int(p * len(lens)))]
print(f"tokens: p50 {q(.5)}, p95 {q(.95)}, p99 {q(.99)}, max {lens[-1]} (max_length {cfg['max_length']})")
print(Counter(r["kind"] for r in train_rows), "clean:", sum(r["clean"] for r in train_rows))
print(json.dumps(train_rows[0]["messages"][1:], indent=1, ensure_ascii=False)[:1500])

## 3. Zero-shot baseline (base model, no adapter)

In [ ]:
eval_rows = sc.eval_rows(Path(DATA), EVAL_N, cfg["seed"])
sc.seed_all(cfg["seed"])
tok0, base = sc.load_base(cfg, sc.precision())
base_out = sc.predict(base, tok0, eval_rows)
base_metrics = sc.quick_metrics(eval_rows, base_out)
print(json.dumps(base_metrics, indent=1))
for r, o in list(zip(eval_rows, base_out))[:3]:
    print("---", r["input"]["unresolved"], "\nTARGET", json.dumps(r["target"], ensure_ascii=False), "\nBASE  ", o[:300])
del base; torch.cuda.empty_cache()

## 4. Train (LoRA, early stopping on val loss)

In [ ]:
summary = sc.train(cfg, Path(DATA), Path(OUT))
print(f"best val loss {summary['best_val_loss']:.4f} after {summary['steps']} steps, {summary['minutes']:.1f} min")
for h in summary["history"]:
    print(f"step {h['step']:4d}  epoch {h['epoch']}  train {h['train_loss']:.4f}  val {h['val_loss']:.4f}")

## 5. Best adapter: 10 val predictions and the same metrics as the baseline

In [ ]:
from peft import PeftModel
torch.cuda.empty_cache()
tok1, base = sc.load_base(cfg, sc.precision())
model = PeftModel.from_pretrained(base, f"{OUT}/adapter")
lora_out = sc.predict(model, tok1, eval_rows)
lora_metrics = sc.quick_metrics(eval_rows, lora_out)
print(json.dumps({"zero_shot": base_metrics, "lora": lora_metrics}, indent=1))
for r, o in list(zip(eval_rows, lora_out))[:10]:
    print(f"=== {r['id']} {r['category']} ({r['kind']})  prompt: {r['input']['prompt'][:150]}")
    print("unresolved:", r["input"]["unresolved"])
    print("target:    ", json.dumps(r["target"], ensure_ascii=False))
    print("predicted: ", o)

## 6. Save the adapter zip to Drive

In [ ]:
import shutil, hashlib, time
Path(OUT, "adapter", "val_predictions.json").write_text(json.dumps(
    [{"id": r["id"], "kind": r["kind"], "target": r["target"], "zero_shot": b, "lora": l}
     for r, b, l in zip(eval_rows, base_out, lora_out)], indent=1, ensure_ascii=False))
Path(OUT, "adapter", "quick_metrics.json").write_text(json.dumps(
    {"zero_shot": base_metrics, "lora": lora_metrics, "data_manifest": manifest}, indent=1))
os.makedirs(OUT_DRIVE, exist_ok=True)
archive = shutil.make_archive("/content/stage_c_adapter_v1", "zip", OUT, "adapter")
shutil.copy(archive, f"{OUT_DRIVE}/stage_c_adapter_v1.zip")
digest = hashlib.sha256(open(archive, "rb").read()).hexdigest()
print(f"saved {OUT_DRIVE}/stage_c_adapter_v1.zip  ({os.path.getsize(archive) / 1e6:.1f} MB)  sha256 {digest}")